# Experiment 1 — Subject-level statistical analysis (Wilson CI + bootstrap CI)

This notebook reads the subject-level outputs saved by `Proposed_model_subject_fulltest.ipynb` (Experiment 1) and produces paper-ready uncertainty estimates:

- **Subject Accuracy**: Wilson 95% CI (binomial over subjects)
- **Subject Macro-F1 / Macro-AUC (OvR)**: 95% bootstrap CI (resample subjects, stratified by true class)

**Input**
- `Results/AL_MODELS/subject_fulltest_subject_level_details.csv`

**Outputs**
- `Results/AL_MODELS/experiment1_subject_stats_per_seed.csv` (one row per init seed)
- `Results/AL_MODELS/experiment1_subject_stats_summary.csv` (one row per model/config)

**Note**
- Mean±std are computed over all init seeds.
- CIs (Wilson/Bootstrap) are computed on a single representative init seed per setting (default: init seed 0) to keep runtime reasonable.


In [5]:
from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score, roc_auc_score

# Resolve project root (Jupyter often runs with cwd=code/)
PROJECT_ROOT = Path.cwd()
for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (p / "Results" / "AL_MODELS").exists():
        PROJECT_ROOT = p
        break

DETAILS_CSV = PROJECT_ROOT / "Results" / "AL_MODELS" / "subject_fulltest_subject_level_details.csv"
OUT_PER_SEED = PROJECT_ROOT / "Results" / "AL_MODELS" / "experiment1_subject_stats_per_seed.csv"
OUT_SUMMARY = PROJECT_ROOT / "Results" / "AL_MODELS" / "experiment1_subject_stats_summary.csv"

# Bootstrap settings
BOOTSTRAP_ITERS = 2000
CONF_LEVEL = 0.95

if not DETAILS_CSV.exists():
    raise FileNotFoundError(f"Missing input CSV: {DETAILS_CSV}")

df = pd.read_csv(DETAILS_CSV)
required_cols = {
    "model",
    "config",
    "smote",
    "merge",
    "split_seed",
    "init_seed",
    "subject_id",
    "n_utts",
    "y_true",
    "y_pred",
    "prob_0",
    "prob_1",
    "prob_2",
}
missing = sorted(required_cols - set(df.columns))
if missing:
    raise ValueError(f"DETAILS_CSV missing columns: {missing}")

def _to_bool(v):
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    s = str(v).strip().lower()
    if s in {"true", "1", "yes", "y", "t"}:
        return True
    if s in {"false", "0", "no", "n", "f"}:
        return False
    return bool(v)

# Normalize dtypes (important if CSV stores True/False as strings)
df["smote"] = df["smote"].map(_to_bool)
df["merge"] = df["merge"].map(_to_bool)
df["split_seed"] = df["split_seed"].astype(int)
df["init_seed"] = df["init_seed"].astype(int)
df["y_true"] = df["y_true"].astype(int)
df["y_pred"] = df["y_pred"].astype(int)

CLASS_LABELS = [0, 1, 2]
PROB_COLS = ["prob_0", "prob_1", "prob_2"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded: {DETAILS_CSV}  shape={df.shape}")
print(f"Will write: {OUT_PER_SEED}")
print(f"Will write: {OUT_SUMMARY}")


Project root: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta
Loaded: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/subject_fulltest_subject_level_details.csv  shape=(7800, 15)
Will write: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/experiment1_subject_stats_per_seed.csv
Will write: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/experiment1_subject_stats_summary.csv


In [6]:
def wilson_ci(k: int, n: int, conf_level: float = 0.95) -> tuple[float, float]:
    """Wilson score interval for a binomial proportion."""
    if n <= 0:
        return (float("nan"), float("nan"))
    if k < 0 or k > n:
        raise ValueError(f"Invalid k={k} for n={n}")

    # z for two-sided normal CI
    from math import sqrt
    from statistics import NormalDist

    alpha = 1.0 - conf_level
    z = NormalDist().inv_cdf(1.0 - alpha / 2.0)
    phat = k / n

    denom = 1.0 + (z**2) / n
    center = (phat + (z**2) / (2.0 * n)) / denom
    half = (z * sqrt((phat * (1.0 - phat) + (z**2) / (4.0 * n)) / n)) / denom
    lo = max(0.0, center - half)
    hi = min(1.0, center + half)
    return (lo, hi)


def stratified_bootstrap_ci(
    y_true: np.ndarray,
    y_pred: np.ndarray | None,
    y_prob: np.ndarray | None,
    metric: str,
    iters: int,
    rng: np.random.Generator,
    conf_level: float = 0.95,
) -> tuple[float, float]:
    """Stratified bootstrap CI by resampling subjects within each true class."""
    y_true = np.asarray(y_true)
    if metric in {"f1"}:
        if y_pred is None:
            raise ValueError("y_pred required for f1")
        y_pred = np.asarray(y_pred)
    if metric in {"auc"}:
        if y_prob is None:
            raise ValueError("y_prob required for auc")
        y_prob = np.asarray(y_prob)

    per_class_indices = {c: np.flatnonzero(y_true == c) for c in CLASS_LABELS}
    for c, idx in per_class_indices.items():
        if len(idx) == 0:
            # If a class is absent in test subjects, CI isn't meaningful.
            return (float("nan"), float("nan"))

    values: list[float] = []
    for _ in range(iters):
        sampled = []
        for c in CLASS_LABELS:
            idx = per_class_indices[c]
            sampled.append(rng.choice(idx, size=len(idx), replace=True))
        bs_idx = np.concatenate(sampled)

        yt = y_true[bs_idx]
        if metric == "f1":
            yp = y_pred[bs_idx]
            v = f1_score(yt, yp, average="macro", labels=CLASS_LABELS, zero_division=0)
        elif metric == "auc":
            ys = y_prob[bs_idx]
            # Multiclass OvR macro AUC
            v = roc_auc_score(yt, ys, multi_class="ovr", average="macro", labels=CLASS_LABELS)
        else:
            raise ValueError(f"Unknown metric: {metric}")
        values.append(float(v))

    alpha = 1.0 - conf_level
    lo = float(np.quantile(values, alpha / 2.0))
    hi = float(np.quantile(values, 1.0 - alpha / 2.0))
    return (lo, hi)


rows = []
group_cols = ["model", "config", "smote", "merge", "split_seed", "init_seed"]

# 1) Fast per-seed metrics (no bootstrap here)
for key, g in df.groupby(group_cols, sort=True):
    model, config, smote, merge, split_seed, init_seed = key

    y_true = g["y_true"].to_numpy(dtype=int)
    y_pred = g["y_pred"].to_numpy(dtype=int)
    y_prob = g[PROB_COLS].to_numpy(dtype=float)

    n_subj = int(len(y_true))
    k_correct = int(np.sum(y_true == y_pred))

    subject_acc = k_correct / n_subj if n_subj else float("nan")
    subject_f1 = float(
        f1_score(y_true, y_pred, average="macro", labels=CLASS_LABELS, zero_division=0)
    )
    subject_auc = float(
        roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro", labels=CLASS_LABELS)
    )
    acc_ci_lo, acc_ci_hi = wilson_ci(k_correct, n_subj, conf_level=CONF_LEVEL)

    rows.append(
        {
            "model": model,
            "config": config,
            "smote": bool(smote),
            "merge": bool(merge),
            "split_seed": int(split_seed),
            "init_seed": int(init_seed),
            "n_test_subjects": n_subj,
            "k_correct": k_correct,
            "subject_acc": subject_acc,
            "subject_acc_ci_low": acc_ci_lo,
            "subject_acc_ci_high": acc_ci_hi,
            "subject_f1_macro": subject_f1,
            "subject_auc_ovr_macro": subject_auc,
        }
    )

per_seed = pd.DataFrame(rows)
OUT_PER_SEED.parent.mkdir(parents=True, exist_ok=True)
per_seed.to_csv(OUT_PER_SEED, index=False)
print(f"Saved: {OUT_PER_SEED}  rows={len(per_seed)}")

# 2) Summary mean±std across init seeds + k_correct distribution
sum_cols = ["model", "config", "smote", "merge", "split_seed"]

def _k_dist(s: pd.Series) -> str:
    d = s.value_counts().sort_index().to_dict()
    return json.dumps({int(k): int(v) for k, v in d.items()}, ensure_ascii=False)

summary = (
    per_seed.groupby(sum_cols, sort=True)
    .agg(
        n_init_seeds=("init_seed", "nunique"),
        n_test_subjects=("n_test_subjects", "first"),
        k_correct_dist=("k_correct", _k_dist),
        subject_acc_mean=("subject_acc", "mean"),
        subject_acc_std=("subject_acc", "std"),
        subject_f1_mean=("subject_f1_macro", "mean"),
        subject_f1_std=("subject_f1_macro", "std"),
        subject_auc_mean=("subject_auc_ovr_macro", "mean"),
        subject_auc_std=("subject_auc_ovr_macro", "std"),
    )
    .reset_index()
)

# 3) Add Wilson/Bootstrap CIs using a representative init seed (default: seed 0)
rep_seed_by_setting = (
    per_seed.groupby(sum_cols, sort=True)["init_seed"].min().rename("rep_init_seed").reset_index()
)
summary = summary.merge(rep_seed_by_setting, on=sum_cols, how="left")

ci_rows = []
for _, r in summary.iterrows():
    model = r["model"]
    config = r["config"]
    smote = bool(r["smote"])
    merge = bool(r["merge"])
    split_seed = int(r["split_seed"])
    rep_init_seed = int(r["rep_init_seed"])

    g = df[
        (df["model"] == model)
        & (df["config"] == config)
        & (df["smote"] == smote)
        & (df["merge"] == merge)
        & (df["split_seed"] == split_seed)
        & (df["init_seed"] == rep_init_seed)
    ]
    y_true = g["y_true"].to_numpy(dtype=int)
    y_pred = g["y_pred"].to_numpy(dtype=int)
    y_prob = g[PROB_COLS].to_numpy(dtype=float)
    n_subj = int(len(y_true))
    k_correct = int(np.sum(y_true == y_pred))

    acc_ci_lo, acc_ci_hi = wilson_ci(k_correct, n_subj, conf_level=CONF_LEVEL)
    rng = np.random.default_rng(int(split_seed) * 1000 + int(rep_init_seed))
    f1_ci_lo, f1_ci_hi = stratified_bootstrap_ci(
        y_true=y_true,
        y_pred=y_pred,
        y_prob=None,
        metric="f1",
        iters=BOOTSTRAP_ITERS,
        rng=rng,
        conf_level=CONF_LEVEL,
    )
    auc_ci_lo, auc_ci_hi = stratified_bootstrap_ci(
        y_true=y_true,
        y_pred=None,
        y_prob=y_prob,
        metric="auc",
        iters=BOOTSTRAP_ITERS,
        rng=rng,
        conf_level=CONF_LEVEL,
    )

    ci_rows.append(
        {
            "model": model,
            "config": config,
            "smote": smote,
            "merge": merge,
            "split_seed": split_seed,
            "rep_init_seed": rep_init_seed,
            "rep_n_test_subjects": n_subj,
            "rep_k_correct": k_correct,
            "subject_acc_ci_low": acc_ci_lo,
            "subject_acc_ci_high": acc_ci_hi,
            "subject_f1_ci_low": f1_ci_lo,
            "subject_f1_ci_high": f1_ci_hi,
            "subject_auc_ci_low": auc_ci_lo,
            "subject_auc_ci_high": auc_ci_hi,
            "bootstrap_iters": int(BOOTSTRAP_ITERS),
        }
    )

ci_df = pd.DataFrame(ci_rows)
summary = summary.merge(ci_df, on=sum_cols + ["rep_init_seed"], how="left")

summary.to_csv(OUT_SUMMARY, index=False)
print(f"Saved: {OUT_SUMMARY}  rows={len(summary)}")

# Print a compact view (paper-friendly)
view = summary[[
    "model",
    "config",
    "subject_acc_mean",
    "subject_acc_std",
    "subject_acc_ci_low",
    "subject_acc_ci_high",
    "subject_f1_mean",
    "subject_f1_std",
    "subject_f1_ci_low",
    "subject_f1_ci_high",
    "subject_auc_mean",
    "subject_auc_std",
    "subject_auc_ci_low",
    "subject_auc_ci_high",
    "n_test_subjects",
    "n_init_seeds",
    "rep_init_seed",
    "k_correct_dist",
]]

pd.set_option("display.max_colwidth", 120)
display(view.sort_values(["model", "config"]))

# Extra sanity check: show where std prints as ~0 but mean != 1
near_zero = 1e-6
suspect = summary[(summary["subject_auc_std"].fillna(0) < near_zero) & (summary["subject_auc_mean"] < 0.999999)]
if len(suspect) > 0:
    print("\nCases where subject_auc_std ~ 0 but subject_auc_mean != 1 (expected when AUC is identical across seeds):")
    display(suspect[["model", "config", "subject_auc_mean", "subject_auc_std", "k_correct_dist"]])


Saved: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/experiment1_subject_stats_per_seed.csv  rows=600
Saved: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/experiment1_subject_stats_summary.csv  rows=20


,model,config,subject_acc_mean,subject_acc_std,subject_acc_ci_low,subject_acc_ci_high,subject_f1_mean,subject_f1_std,subject_f1_ci_low,subject_f1_ci_high,subject_auc_mean,subject_auc_std,subject_auc_ci_low,subject_auc_ci_high,n_test_subjects,n_init_seeds,rep_init_seed,k_correct_dist
0,Hubert,acoustic,0.971795,0.037703,0.666860,0.986290,0.966154,0.045243,0.600000,1.000000,1.000000,0.000000,1.000000,1.000000,13,30,0,"{""12"": 11, ""13"": 19}"
1,Hubert,fullspec,0.920513,0.042773,0.666860,0.986290,0.920455,0.041599,0.777778,1.000000,0.999259,0.002819,1.000000,1.000000,13,30,0,"{""11"": 5, ""12"": 21, ""13"": 4}"
2,Hubert,last3,0.843590,0.014044,0.577654,0.956742,0.819349,0.012049,0.545455,1.000000,0.958765,0.011620,0.881481,1.000000,13,30,0,"{""10"": 1, ""11"": 29}"
3,Hubert,middle3,0.920513,0.014044,0.666860,0.986290,0.919675,0.013160,0.777778,1.000000,0.987778,0.003390,0.955556,1.000000,13,30,0,"{""11"": 1, ""12"": 29}"
4,Hubert,stability,0.923077,0.000000,0.666860,0.986290,0.922078,0.000000,0.777778,1.000000,0.984444,0.007496,0.955556,1.000000,13,30,0,"{""12"": 30}"
5,data2vec,acoustic,0.917949,0.019516,0.666860,0.986290,0.920830,0.022490,0.733333,1.000000,0.990741,0.000000,0.944444,1.000000,13,30,0,"{""11"": 2, ""12"": 28}"
6,data2vec,fullspec,0.902564,0.034598,0.577654,0.956742,0.903101,0.039870,0.535714,1.000000,0.984127,0.003008,0.896825,1.000000,13,30,0,"{""11"": 8, ""12"": 22}"
7,data2vec,last3,0.700000,0.023471,0.423693,0.873193,0.643939,0.023116,0.384127,0.896296,0.905053,0.013510,0.781217,0.977778,13,30,0,"{""9"": 27, ""10"": 3}"
8,data2vec,middle3,0.920513,0.014044,0.666860,0.986290,0.929450,0.017581,0.797980,1.000000,0.995459,0.007765,1.000000,1.000000,13,30,0,"{""11"": 1, ""12"": 29}"
9,data2vec,stability,0.923077,0.000000,0.666860,0.986290,0.932660,0.000000,0.797980,1.000000,0.995370,0.005831,1.000000,1.000000,13,30,0,"{""12"": 30}"



Cases where subject_auc_std ~ 0 but subject_auc_mean != 1 (expected when AUC is identical across seeds):


,model,config,subject_auc_mean,subject_auc_std,k_correct_dist
5,data2vec,acoustic,0.990741,0.0,"{""11"": 2, ""12"": 28}"
